# SafeScan — Notebook 3: LLM Layer (Allergen Detection + Nutrition Health Score)
### Front/Back split — v2

**What changed from your original single-image version:**
1. **Front image** → OCR + ResNet50 → classifier → product category (this is what Notebook 2's model was actually trained to do — front covers carry the clearest brand/logo signal).
2. **Back image** → OCR → Gemini allergen check + nutrition score (ingredient and nutrition panels live on the back).
3. Fixed a bug: `preprocessors.pkl` key names now match what Notebook 2 actually saves (`vectorizer`, `scaler_img`, `scaler_text`, `label_to_int`, `unique_labels`) — the original names (`tfidf_vectorizer`, `scaler_image`, `label_map`) don't exist in your saved file and would throw a `KeyError`.
4. Swapped `gemini-2.5-flash` (deprecated) for the rolling alias `gemini-flash-latest`.
5. Added retry-with-backoff around Gemini calls, since the free tier occasionally returns `503 UNAVAILABLE` under high demand — this was the error you hit.
6. Install cell no longer force-upgrades `tensorflow`/`scikit-learn` — Colab's preinstalled versions already match what trained your model; upgrading risked breaking `load_model()`.

No changes to Notebook 2 or the trained model — this is purely an inference-flow update.

## Step 1 — Install packages
Run this first, every session (Colab resets each time).

**Note:** installs only what Colab doesn't already have. We deliberately don't force-upgrade `tensorflow` or `scikit-learn` — Colab's preinstalled versions already match what trained your model.

In [ ]:
!pip install -q google-genai easyocr gdown
print("Packages installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 19.3 MB/s eta 0:00:00
Packages installed.


## Step 2 — Get your trained model files back into this notebook

Same gdown trick as before — file IDs already filled in below.

In [ ]:
import gdown

MODEL_FILE_ID = "1kM577z4OiSRBtOsZ_HqB2H8gyMusxOMG"
PREPROCESSORS_FILE_ID = "1uwO1n9JE4mBl5S4tTNAmZSvh8d9Qf8T1"

gdown.download(id=MODEL_FILE_ID, output="safescan_classifier.h5", quiet=False)
gdown.download(id=PREPROCESSORS_FILE_ID, output="preprocessors.pkl", quiet=False)

print("Downloaded model + preprocessors.")


Downloading...
From: https://drive.google.com/uc?id=1kM577z4OiSRBtOsZ_HqB2H8gyMusxOMG
To: /content/safescan_classifier.h5
100%|██████████| 17.8M/17.8M [00:00<00:00, 30.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1uwO1n9JE4mBl5S4tTNAmZSvh8d9Qf8T1
To: /content/preprocessors.pkl
100%|██████████| 80.3k/80.3k [00:00<00:00, 1.41MB/s]

Downloaded model + preprocessors.


**If gdown fails** with a "cannot retrieve" error, the file isn't shared as "Anyone with the link" — fix the Drive sharing setting and re-run this cell.

## Step 3 — Load the model and preprocessors into memory

In [ ]:
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model

classifier = load_model("safescan_classifier.h5")

with open("preprocessors.pkl", "rb") as f:
    preproc = pickle.load(f)

# Fixed: these now match what Notebook 2 (Cell 9) actually saves.
vectorizer     = preproc["vectorizer"]
scaler_img     = preproc["scaler_img"]
scaler_text    = preproc["scaler_text"]
label_to_int   = preproc["label_to_int"]
unique_labels  = preproc["unique_labels"]
int_to_label   = {v: k for k, v in label_to_int.items()}

print("Model + preprocessors loaded.")
print("Categories:", unique_labels)


Model + preprocessors loaded.
Categories: ['ALMONDS_NUTS', 'BISCUITS', 'BUTTER', 'CEREAL', 'CHEESE', 'CHIPS_SNACKS', 'CHOCOLATE_CANDY', 'HONEY', 'MILK', 'NOODLES_PASTA', 'OIL', 'PICKLE', 'PULSES_DAL', 'RICE', 'SALT', 'SAUCE_KETCHUP', 'SOFTDRINK_JUICE', 'SPICES', 'SUGAR', 'TEA', 'WATER']


## Step 4 — Set up Gemini API access

**Security note:** never paste your API key directly into a code cell you might share or upload to GitHub. The cell below uses `getpass` so the key is typed hidden and never saved in the notebook file.

Get a free key at: https://aistudio.google.com/apikey

⚠️ If you've ever pasted this key in a chat message anywhere (including to me), treat it as compromised — regenerate a fresh one before using it here.

In [ ]:
import getpass
from google import genai

GEMINI_API_KEY = getpass.getpass("Paste your Gemini API key (hidden): ")
client = genai.Client(api_key=GEMINI_API_KEY)

# Rolling alias — avoids repeat breakage when Google retires specific model versions
# (gemini-2.5-flash, used previously, has been deprecated per your project notes).
GEMINI_MODEL = "gemini-flash-latest"

print("Gemini client ready.")


Paste your Gemini API key (hidden): ··········
Gemini client ready.


### Retry wrapper for Gemini calls
Free-tier Gemini occasionally returns `503 UNAVAILABLE` under high demand (this is what crashed your last run) — not a bug in your code. This wrapper retries a few times with a short delay before giving up.

In [ ]:
import time

def call_gemini_with_retry(prompt, retries=3, delay=5):
    """
    Calls Gemini with the given text prompt, retrying on transient server errors
    (like 503 UNAVAILABLE) before giving up.
    """
    last_error = None
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        except Exception as e:
            last_error = e
            if attempt < retries - 1:
                print(f"Gemini busy (attempt {attempt + 1}/{retries}), retrying in {delay}s...")
                time.sleep(delay)
    raise last_error


## Step 5 — Define the user's allergy profile

Plain list — edit to whatever allergens you want to demo/test with.

In [ ]:
# EDIT THIS — the allergens this particular user is avoiding
ALLERGY_PROFILE = ["peanuts", "milk", "gluten"]

print("Allergy profile set to:", ALLERGY_PROFILE)


Allergy profile set to: ['peanuts', 'milk', 'gluten']


## Step 6 — OCR extraction

Same function used for both front and back images.

In [ ]:
import easyocr
import re

ocr_reader = easyocr.Reader(['en'], gpu=True)

def clean_text(raw_text):
    text = raw_text.lower()
    text = re.sub(r'[^a-z0-9\s%.,]', ' ', text)
    tokens = [t for t in text.split() if len(t) >= 2]
    return " ".join(tokens)

def extract_ocr_text(image_path):
    results = ocr_reader.readtext(image_path, detail=0)
    raw_text = " ".join(results)
    return clean_text(raw_text)

print("OCR function ready.")


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteOCR function ready.


## Step 7 — Classification function (runs on the FRONT image only)

Predicts which of the categories the product belongs to, using the same image+text fusion approach as Notebook 2 — a single concatenated (2048 + 500)-dim vector fed to one classifier input.

In [ ]:
import numpy as np
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image as keras_image

resnet_base = ResNet50(weights="imagenet", include_top=False, pooling="avg")

def get_image_features(image_path):
    img = keras_image.load_img(image_path, target_size=(224, 224))
    x = keras_image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    features = resnet_base.predict(x, verbose=0)
    return features  # shape (1, 2048)

def classify_product(front_image_path, front_ocr_text):
    img_feat = get_image_features(front_image_path)
    img_feat_scaled = scaler_img.transform(img_feat)

    text_feat = vectorizer.transform([front_ocr_text]).toarray()
    text_feat_scaled = scaler_text.transform(text_feat)

    fused = np.concatenate([img_feat_scaled, text_feat_scaled], axis=1)
    probs = classifier.predict(fused, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    confidence = float(probs[pred_idx])

    return int_to_label[pred_idx], confidence

print("Classification function ready.")


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Classification function ready.


## Step 8 — Gemini functions: allergen check + nutrition score (run on the BACK image's OCR text)

Two small, focused prompts — one job each. Both now go through the retry wrapper.

In [ ]:
def check_allergens(back_ocr_text, allergy_profile):
    prompt = f"""You are a food safety assistant. Below is raw OCR text extracted
from a grocery product's ingredient label (it may contain noise/typos from OCR).

OCR TEXT:
{back_ocr_text}

USER'S ALLERGENS TO AVOID: {", ".join(allergy_profile)}

Task:
1. List any ingredients in the OCR text that conflict with the user's allergens.
2. If none are found, clearly say the product appears safe based on the visible text,
   but note that OCR can miss text, so the user should still check the physical label.
3. Keep the explanation short — 2 to 4 sentences, plain language, no medical jargon.

Respond in this exact format:
VERDICT: <SAFE / CONFLICT FOUND / UNCERTAIN>
DETAILS: <your explanation>"""

    response = call_gemini_with_retry(prompt)
    return response.text


def get_nutrition_score(back_ocr_text):
    prompt = f"""You are a nutrition assistant. Below is raw OCR text extracted from
a grocery product's nutrition facts panel (it may contain noise/typos from OCR).

OCR TEXT:
{back_ocr_text}

Task:
1. If nutrition facts (calories, sugar, fat, sodium, etc.) are identifiable in the text,
   give the product a health score out of 10 (10 = very healthy, 1 = very unhealthy).
2. If no nutrition facts are visible in the OCR text, say so clearly instead of guessing.
3. Give a 2 to 3 sentence plain-language reason for the score.

Respond in this exact format:
SCORE: <X/10 or "Not available">
REASON: <your explanation>"""

    response = call_gemini_with_retry(prompt)
    return response.text

print("Gemini functions ready.")


Gemini functions ready.


## Step 9 — Full pipeline (front/back)

Takes two image paths — front for classification, back for allergen + nutrition analysis — and prints one combined report.

In [ ]:
def run_safescan(front_image_path, back_image_path, allergy_profile=ALLERGY_PROFILE):
    print(f"Front image: {front_image_path}")
    print(f"Back image : {back_image_path}")

    front_ocr_text = extract_ocr_text(front_image_path)
    category, confidence = classify_product(front_image_path, front_ocr_text)

    back_ocr_text = extract_ocr_text(back_image_path)
    allergen_result = check_allergens(back_ocr_text, allergy_profile)
    nutrition_result = get_nutrition_score(back_ocr_text)

    print("\n" + "="*50)
    print("SAFESCAN REPORT")
    print("="*50)
    print(f"Predicted category : {category}  (confidence: {confidence:.1%})")
    print(f"Front OCR text (raw): {front_ocr_text[:150]}{'...' if len(front_ocr_text) > 150 else ''}")
    print(f"Back OCR text (raw) : {back_ocr_text[:200]}{'...' if len(back_ocr_text) > 200 else ''}")
    print("-"*50)
    print("ALLERGEN CHECK")
    print(allergen_result)
    print("-"*50)
    print("NUTRITION HEALTH SCORE")
    print(nutrition_result)
    print("="*50)

    return {
        "category": category,
        "confidence": confidence,
        "front_ocr_text": front_ocr_text,
        "back_ocr_text": back_ocr_text,
        "allergen_result": allergen_result,
        "nutrition_result": nutrition_result,
    }

print("Pipeline ready. Upload front + back images in the next cell to test it.")


Pipeline ready. Upload front + back images in the next cell to test it.


## Step 10 — Test it on a real product

Run this cell — you'll be prompted to upload twice: **front** first, then **back**.

In [ ]:
from google.colab import files

print("Upload the FRONT image:")
front_uploaded = files.upload()
front_image_path = list(front_uploaded.keys())[0]

print("\nUpload the BACK image:")
back_uploaded = files.upload()
back_image_path = list(back_uploaded.keys())[0]

result = run_safescan(front_image_path, back_image_path)


Upload the FRONT image:


Saving IMG_7334.jpg to IMG_7334 (2).jpg

Upload the BACK image:


Saving IMG_20190419_171130.jpg to IMG_20190419_171130 (2).jpg
Front image: IMG_7334 (2).jpg
Back image : IMG_20190419_171130 (2).jpg
Gemini busy (attempt 1/3), retrying in 5s...

SAFESCAN REPORT
Predicted category : BISCUITS  (confidence: 99.5%)
Front OCR text (raw): artificial flavouring subs inacool hygienic and day pllace transfer contente to cleah airtight contnner once caened britannia nutci choice digestive h...
Back OCR text (raw) : 8812042800, 100 years of britahhia mail feedbackobritindiacom nutrition 1oog product ac0n information ivgredients carbohydrates 689 rmfaved wheat mlour whc le wheat of which sugars 14.59 edible veieta...
--------------------------------------------------
ALLERGEN CHECK
VERDICT: CONFLICT FOUND
DETAILS: This product contains ingredients that conflict with your gluten and milk allergies. The label lists several wheat and gluten sources, including wheat flour, whole wheat flour, wheat bran, and malt extract. Additionally, it contains "mck solids" (an OCR

## Notes / troubleshooting

- **"No module named google.genai"** → re-run the Step 1 install cell.
- **Gemini call errors (401/403)** → your API key is wrong or expired; regenerate it at https://aistudio.google.com/apikey and re-run Step 4.
- **Gemini `503 UNAVAILABLE`** → transient server overload, not your code. The retry wrapper in Step 4 handles a few automatic retries; if it still fails, wait a minute and re-run Step 10.
- **OCR text looks empty or garbled** → try a clearer, well-lit photo, or crop closer to the ingredients/nutrition panel before uploading (this mainly affects the back image).
- **classify_product output looks wrong** → double-check the key names in `preprocessors.pkl` match Step 3 (`vectorizer`, `scaler_img`, `scaler_text`, `label_to_int`, `unique_labels`).
- **Next step after this notebook**: wrap `run_safescan()` in a FastAPI `/predict` endpoint so a frontend can call it — that's the next unstarted piece of the project.